In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../data/online_shoppers.csv")

In [3]:
data.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [4]:
df = data.drop_duplicates()
df.shape

(12205, 18)

In [5]:
y = df['Revenue']
X= df.drop(columns = 'Revenue')


In [6]:
from sklearn.model_selection import train_test_split

In [7]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify= y)

In [8]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((9764, 17), (2441, 17), (9764,), (2441,))

In [9]:
y_train.mean()

np.float64(0.15628840639082342)

In [10]:
y_test.mean()

np.float64(0.15649324047521507)

## Split
total size after duplicated drop = (12205,18)
test rate = 0.2
Purchase rate: train 15.6%, test 15.6% (stratified)

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

In [12]:
dummy = DummyClassifier(strategy = 'most_frequent')
dummy.fit(X_train,y_train)
predictions = dummy.predict(X_train)

print(f" Accuracy : {accuracy_score(y_train, predictions):.3f} \n Recall : {recall_score(y_train,predictions)}")


 Accuracy : 0.844 
 Recall : 0.0


## Dummy baseline
- The dummy (most_frequent) always predicts majority class for every session.
- Accuracy = 84.4% because 84.4% of sessions are non-buyers, so "always no" is right that often.
- Recall = 0.0% because it never predicts minority class, so it catches none of the real buyers.
- Conclusion: accuracy is misleading on imbalanced data; we use PR-AUC instead.

## Logistic regression pipeline

In [13]:
X_train.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000,9764.000000
mean,2.342687,82.219938,0.510344,34.950623,32.101803,1207.264687,0.020296,0.041469,5.945606,0.062351,2.130582,2.371876,3.134986,4.059402
std,3.349268,179.502363,1.282728,143.828323,44.817312,1926.389159,0.045234,0.046145,18.860076,0.199959,0.909649,1.734820,2.389418,4.026181
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,8.000000,193.466667,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,9.000000,0.000000,0.000000,18.000000,617.000000,0.002857,0.025000,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,94.725000,0.000000,0.000000,39.000000,1483.650000,0.016667,0.049242,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


In [14]:
numeric_cols = ['Administrative', 'Administrative_Duration','Informational','Informational_Duration','ProductRelated','ProductRelated_Duration','BounceRates','ExitRates','PageValues','SpecialDay']
categorical_cols = ['OperatingSystems','Browser','Region','TrafficType','Weekend','Month','VisitorType']
len(numeric_cols + categorical_cols)

17

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [16]:
preprocessor = ColumnTransformer([("num",StandardScaler(),numeric_cols),("category",OneHotEncoder(handle_unknown= 'ignore', sparse_output=False),categorical_cols)])


In [17]:
logreg_pipe = Pipeline([("prep", preprocessor), ("lr_model", LogisticRegression())])

In [18]:
logreg_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('lr_model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[bool](2,)","[False, True]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['Administrative','Administrative_Duration','Informational',..., 'TrafficType','VisitorType','Weekend']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('category', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By speci

In [19]:
logreg_pipe.predict_proba(X_train.head())

array([[0.17766248, 0.82233752],
       [0.42140132, 0.57859868],
       [0.94732716, 0.05267284],
       [0.92838859, 0.07161141],
       [0.95462574, 0.04537426]])

In [20]:
y_train.head()


7556      True
11924     True
3306     False
207      False
4460     False
Name: Revenue, dtype: bool

In [21]:
logreg_pipe.classes_


array([False,  True])

## Logistic regression summary
- Structure: 10 numeric columns are scaled with mean and std, 7 categorical columns are one-hot encoded with 64 features. In total 74 features going into model.
- Unknown categories are ignored (`handle_unknown='ignore'`). Example: TrafficType 12 (1 session) is absent from train.
- Fitted on the training set only.
- `predict_proba` returns 2 columns in the order of `classes_` = [False, True]: column 0 = No purchase, column 1 = Purchase. Each row sums to 1.
- We use column 1 for PR-AUC, because PR-AUC measures how well the model ranks the positive class at the top.
- Sanity check on 5 training rows.

In [22]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits= 5, shuffle= True, random_state= 42)

In [23]:
lr_cv = cross_validate(logreg_pipe, X_train, y_train, cv = cv, scoring= "average_precision")
lr_cv

{'fit_time': array([0.04637074, 0.02464485, 0.03265929, 0.03914785, 0.02996898]),
 'score_time': array([0.00459003, 0.00389814, 0.00392199, 0.00616479, 0.00555706]),
 'test_score': array([0.650092  , 0.59438053, 0.63246486, 0.64764865, 0.65318198])}

In [24]:
print(f" logistic regression + cross validation: \n mean: {lr_cv["test_score"].mean():.3f} \n standard deviation : {lr_cv['test_score'].std():.3f}")

 logistic regression + cross validation: 
 mean: 0.636 
 standard deviation : 0.022


In [25]:
dummy_cv = cross_validate(dummy, X_train, y_train, cv = cv, scoring= "average_precision")
print(dummy_cv['test_score'].mean())
print(dummy_cv['test_score'].std())


0.1562884024577573
0.00019923684608644864


In [26]:
rows = [{'model' : 'Dummy (most frequent)', 'pr_auc_mean' : dummy_cv['test_score'].mean().round(3), 'pr_auc_std':dummy_cv['test_score'].std().round(3)},
        {'model' : 'Logistic Regression', 'pr_auc_mean' : lr_cv["test_score"].mean().round(3), 'pr_auc_std':lr_cv['test_score'].std().round(3)}]

In [27]:
comparison_table = pd.DataFrame(rows)

In [28]:
comparison_table

,model,pr_auc_mean,pr_auc_std
0,Dummy (most frequent),0.156,0.000
1,Logistic Regression,0.636,0.022


## Cross-validation :
- 5-fold stratified CV on the training set, metric = PR-AUC (average precision)
- Logistic regression 0.636 ± 0.022, x4 better than dummy (0.156)

===================================================================================

## Gradient Boosting

In [29]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [30]:
hgb_pipe = Pipeline([("prep", preprocessor), ("hgb_model", HistGradientBoostingClassifier(random_state= 42))])

In [31]:
hgb_cv = cross_validate(hgb_pipe, X_train, y_train, cv = cv, scoring= "average_precision")

In [32]:
print(hgb_cv['test_score'].mean())
print(hgb_cv['test_score'].std())

0.7423994655620276
0.022444269839407883


In [33]:
rows.append({'model' : 'Gradient boosting', 'pr_auc_mean' : hgb_cv['test_score'].mean().round(3), 'pr_auc_std':hgb_cv['test_score'].std().round(3)})

In [34]:
comparison_table = pd.DataFrame(rows)
comparison_table

,model,pr_auc_mean,pr_auc_std
0,Dummy (most frequent),0.156,0.000
1,Logistic Regression,0.636,0.022
2,Gradient boosting,0.742,0.022


## Gradient boosting:
- Same preprocessing and CV folds; only the model changed (HistGradientBoostingClassifier, defaults).
- PR-AUC 0.742 ± 0.022

## Tuning (gradient boosting)

In [35]:
from sklearn.model_selection import GridSearchCV

In [37]:
hgb_pipe['hgb_model']

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing val

In [38]:
grid_params = {
    "hgb_model__learning_rate" : [0.05, 0.1, 0.2],
    "hgb_model__max_leaf_nodes" : [21, 31, 41],
    "hgb_model__min_samples_leaf" : [10, 20, 30]
}

In [39]:
grid_search = GridSearchCV(hgb_pipe, grid_params, cv = cv, scoring= 'average_precision')

In [40]:
grid_search.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'hgb_model__learning_rate': [0.05, 0.1, ...], 'hgb_model__max_leaf_nodes': [21, 31, ...], 'hgb_model__min_samples_leaf': [10, 20, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versi

In [41]:
grid_search.best_params_


{'hgb_model__learning_rate': 0.05,
 'hgb_model__max_leaf_nodes': 21,
 'hgb_model__min_samples_leaf': 20}

In [42]:
grid_search.best_score_

np.float64(0.7568666958555393)

## results:
it increased the score from 0.742 to 0.757 with params of (learning rate = 0.05,
max_leaf_nodes = 21, min_samples_leaf = 20), which is smaller than CV spread 
(±0.022), it's small improvement and could partly be noise.